In [6]:
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(42)

In [ ]:
def bs_put_price(t, S, K, T, r, sigma):
    tau = T - t
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    return (K * np.exp(-r * tau) * norm.cdf(-d2) - S * norm.cdf(-d1))

In [ ]:
def bs_put_delta(t, S, K, T, r, sigma):
    tau = T - t
    d1 = ( np.log(S / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    return norm.cdf(d1) - 1

In [ ]:
def empirical_var(losses, alpha):
    sorted_losses = np.sort(losses)
    n = len(sorted_losses)
    index = int(np.ceil(n * alpha)) - 1
    return sorted_losses[index]

In [16]:
alpha = 0.95
M = 40_000
t = 0.0
dt = 1 / 252
T = 0.292
S0 = 152.51
K = 170
r = 0.00119
sigma = 0.4907
mu = 0.1691
n_puts = 100
J = 10

In [ ]:
P0 = bs_put_price(t, S0, K, T, r, sigma)
h0 = bs_put_delta(t, S0, K, T, r, sigma)
Y0 = 0.0
Y0_new = Y0 - h0 * S0
V0 = (Y0_new + h0 * S0 - P0)

print("Put price:", P0)
print("Put delta:", h0)
print("Money market position:", Y0_new)
print("Initial portfolio value:", V0)

Put price: 27.107728974527646
Put delta: -0.6085548974709577
Money market position: 92.81070741329576
Initial portfolio value: -27.107728974527646


In [ ]:
#1
Z = rng.standard_normal(M)

log_return_sim = ((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
S1_sim = S0 * np.exp(log_return_sim)
P1_sim = bs_put_price(
    t + dt,
    S1_sim,
    K,
    T,
    r,
    sigma
)

Y1 = Y0_new * np.exp(r * dt)
V1_sim = (h0 * S1_sim + Y1 - P1_sim)

losses_1d = -n_puts * (V1_sim - V0)
var_1d = empirical_var(
    losses_1d,
    alpha
)

var_10d_sqrt = np.sqrt(10) * var_1d

print(f"1-day VaR (95%): ${var_1d:,.2f}")
print(f"10-day VaR using sqrt(10): ${var_10d_sqrt:,.2f}")

print(f"Initial put price: ${P0:.4f}")
print(f"Initial put delta: {h0:.6f}")
print(f"Initial stock position for 100 puts: {n_puts * h0:.4f} shares")
print(f"Initial money market position per put: ${Y0_new:.4f}")

1-day VaR (95%): $30.35
10-day VaR using sqrt(10): $95.99
Initial put price: $27.1077
Initial put delta: -0.608555
Initial stock position for 100 puts: -60.8555 shares
Initial money market position per put: $92.8107


In [18]:
J = 10

Z = rng.standard_normal((M, J))

S = np.full(M, S0)
h = np.full(M, h0)
Y_new = np.full(M, -h0 * S0)

for j in range(1, J + 1):

    current_t = t + j * dt

    log_return = (
        (mu - 0.5 * sigma**2) * dt
        + sigma * np.sqrt(dt) * Z[:, j - 1]
    )

    S = S * np.exp(log_return)

    Y = Y_new * np.exp(r * dt)

    P = bs_put_price(
        current_t,
        S,
        K,
        T,
        r,
        sigma
    )

    V = h * S + Y - P

    if j < J:
        h_new = bs_put_delta(
            current_t,
            S,
            K,
            T,
            r,
            sigma
        )

        Y_new = Y + (h - h_new) * S

        h = h_new

losses_10d = -n_puts * (V - V0)

var_10d_sim = empirical_var(
    losses_10d,
    alpha
)

print(f"10-day VaR from time aggregation: ${var_10d_sim:,.2f}")

10-day VaR from time aggregation: $89.49
